In [12]:
class ButterflyDataset(data.Dataset):
    def __init__(self, df, img_dir, transform=None):
        self.img_labels = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform

        self.classes = sorted(self.img_labels['label'].unique())
        self.class_to_idx = {cls_name: idx for idx, cls_name in enumerate(self.classes)}

    def __len__(self):
        return len(self.img_labels)

    def __getitem__(self, idx):
        img_name = self.img_labels.iloc[idx]['filename']
        img_path = os.path.join(self.img_dir, img_name)

        image = Image.open(img_path).convert("RGB")

        label_name = self.img_labels.iloc[idx]['label']
        label_idx = self.class_to_idx[label_name]
        label = torch.tensor(label_idx, dtype=torch.long)

        if self.transform:
            image = self.transform(image)

        return image, label

In [14]:
import kagglehub
import os
import pandas as pd
import torch.utils.data as data
import torchvision.transforms as transforms

BATCH_SIZE = 32
IMAGE_SIZE = 64
path = kagglehub.competition_download('aca-butterflies')

img_dir = os.path.join(path, 'competition', 'train')
df = pd.read_csv(os.path.join(path, 'competition', 'train.csv'))

data_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor()
])

dataset = ButterflyDataset(df=df, img_dir=img_dir, transform=data_transform)

dataloader = data.DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)